# MAIZ_14C · Misma receta de preprocesamiento en todos los ejes · GPU L4 (solo inferencia)

**Entorno: GPU L4.** Pulse **Ejecutar todas**. Tarda alrededor de 1 h.

Al revisar los resultados del 14A encontramos lo siguiente:

- Las 954 imágenes de Pandian2019 que son idénticas byte a byte a PlantVillage reciben recalls de roya muy distintos según la receta de entrada:
  - **Recorte central** (como en el test interno y en Pandian2019): 0,12–0,76.
  - **Redimensionado directo a 224×224** (como en PlantVillage y TOM2024): 0,57–0,93.
- Este notebook evalúa los 30 checkpoints en cada eje (test interno, Pandian2019, PlantVillage, roya de PlantVillage y TOM2024) con **las dos recetas**.
- Así se puede comparar el ranking con una receta común.
- No entrena nada. Antes de calcular, verifica que la receta histórica de cada eje reproduce las predicciones archivadas.

**Resultado:** `MAIZ_14C_Resultados.zip`, en `REVISION_POST_REVIEW_20261029/` y como descarga.

In [ ]:
import shutil,subprocess
if not shutil.which('nvidia-smi'):raise RuntimeError('Este archivo requiere GPU. Selecciónela en Entorno de ejecución > Cambiar tipo.')
p=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
GPU_NAME=p.stdout.strip();print('GPU:',GPU_NAME)
if 'L4' not in GPU_NAME:print('AVISO: no es L4. Puede continuar; la compuerta de concordancia documentará cualquier diferencia.')

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/TesisQ1_Maiz')
if not ROOT.is_dir() or not (ROOT/'01_datasets').is_dir():
    raise FileNotFoundError('No se encuentra TesisQ1_Maiz/01_datasets en Mi unidad.')
for need in ['REVISION_POST_REVIEW_20261029/evaluation_5seeds/prepared/'+d+'/images.tar' for d in ['internal_test','pandian2019','plantvillage','tom2024_primary']]:
    print(('OK   ' if (ROOT/need).exists() else 'FALTA'),need)

In [ ]:
# Paquetes: no se reinstala torch. timm se fija en la versión usada para entrenar.
import importlib.metadata as md, subprocess, sys
def ver(p):
    try:return md.version(p)
    except md.PackageNotFoundError:return None
pk=[]
if ver('timm')!='1.0.15':pk.append('timm==1.0.15')
if ver('grad-cam') is None:pk.append('grad-cam==1.5.4')
if ver('ImageHash') is None:pk.append('imagehash==4.3.1')
if pk:subprocess.run([sys.executable,'-m','pip','install','-q',*pk],check=True)
import torch,torchvision,timm
print('torch',torch.__version__,'| torchvision',torchvision.__version__,'| timm',timm.__version__)

In [ ]:
# Código y referencias de MAIZ_14C (verificados por SHA-256)
import base64,hashlib,io,zipfile,importlib,sys
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='842e645645b8fa403de4a5f381a8518ac0d98b665a551517f816bd58137f9ebe'
CODE=Path('/content/MAIZ14_code');CODE.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(raw)).extractall(CODE)
sys.path.insert(0,str(CODE))
import maiz14,maiz14_run as R,maiz14c as C
importlib.reload(maiz14);importlib.reload(R);importlib.reload(C)
print('Versión:',maiz14.VERSION,'| archivos:',sorted(p.name for p in CODE.iterdir()))

In [ ]:
# Ejecución completa (reanudable: los modelos ya calculados se omiten)
C.run_all(ROOT)

In [ ]:
import pandas as pd
O=ROOT/C.OUT_REL;pd.set_option('display.width',220)
display(pd.read_csv(O/'agreement_gate_14c.csv').describe().loc[['max']])
display(pd.read_csv(O/'ranking_concordance_by_protocol.csv'))
sm=pd.read_csv(O/'architecture_means_by_protocol.csv');display(sm.pivot_table(index=['domain','architecture'],columns='protocol',values='mean').round(4))

In [ ]:
from google.colab import files
files.download('/content/MAIZ_14C_Resultados.zip')